# Intialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Read from Bronze

In [0]:
df=spark.table('workspace.bronze.cust_az12')

# Exploring

In [0]:
print("4️⃣ عينة من البيانات (اضغط على Data Profile):")
display(df.limit(100))

# 1. فحص الهيكل وأنواع البيانات
print("1️⃣ هيكل الجدول وأنواع البيانات (Schema & Data Types):")
df.printSchema()
print("-" * 50)

# 2. فحص التكرار (Duplicates Check)
total_rows = df.count()
distinct_rows = df.distinct().count()
print("2️⃣ فحص التكرار:")
print(f"- إجمالي الصفوف: {total_rows}")
print(f"- الصفوف المكررة: {total_rows - distinct_rows} صف مكرر")
print("-" * 50)

# 3. تقرير القيم المفقودة (Nulls)
print("3️⃣ تقرير القيم المفقودة (Nulls):")
display(df.pandas_api().isnull().sum(axis=0))
print("-" * 50)



# Transformations

## Trimming


In [0]:
trim_exprs = [
    F.trim(F.col(c)).alias(c) if t == 'string' else F.col(c)
    for c, t in df.dtypes
]
df = df.select(*trim_exprs)

## Customer ID Cleanup

In [0]:
df = df.withColumn(
    "cid",
    F.when(col("cid").startswith("NAS"),
           F.substring(col("cid"), 4, F.length(col("cid"))))
     .otherwise(col("cid"))
)


## Birthdate Validation


In [0]:
df = df.withColumn(
    "bdate",
    F.when(col("bdate") > F.current_date(), None)
     .otherwise(col("bdate"))
)

## Gender Normalization


In [0]:
df = df.withColumn(
    "gen",
    F.when(F.upper(col("gen")).isin("F", "FEMALE"), "Female")
     .when(F.upper(col("gen")).isin("M", "MALE"), "Male")
     .otherwise("n/a")
)

## Renaming Columns

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity checks of dataframe

In [0]:
df.limit(10).display()

# Write Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customers")

## Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.erp_customers LIMIT 10